# 1. Set up

In [0]:
from pyspark.sql import functions as F, types as T, Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, PCA


In [0]:
dbutils.widgets.text("contract_year", "")
dbutils.widgets.text("run_ts", "")  # optional override e.g. 2026-01-02 10:00:00

contract_year = dbutils.widgets.get("contract_year")
run_ts = dbutils.widgets.get("run_ts")


In [0]:
def sanitize_numeric(df, cols, fill_value=0.0):
    """
    Cast to double; convert NaN/Inf to null; fill null.
    Prevents VectorAssembler errors later.
    """
    for c in cols:
        if c not in df.columns:
            continue
        df = df.withColumn(c, F.col(c).cast("double"))
        df = df.withColumn(
            c,
            F.when(F.col(c).isNull(), None)
             .when(F.isnan(F.col(c)), None)
             .when(F.col(c) == float("inf"), None)
             .when(F.col(c) == float("-inf"), None)
             .otherwise(F.col(c))
        )
    return df.fillna({c: fill_value for c in cols if c in df.columns})

def zscore(df, col):
    """
    Add zscore column: (x - mean)/stddev. Handles std=0.
    """
    stats = df.agg(F.mean(col).alias("m"), F.stddev(col).alias("s")).collect()[0]
    m, s = stats["m"], stats["s"]
    if m is None:
        return df.withColumn(f"{col}_z", F.lit(0.0))
    if s is None or s == 0:
        return df.withColumn(f"{col}_z", (F.col(col) - F.lit(m)) * 0.0)
    return df.withColumn(f"{col}_z", (F.col(col) - F.lit(m)) / F.lit(s))

def quantile_bucket(col, q1, q2):
    """
    3-bucket: 0/1/2
    """
    return (
        F.when(F.col(col) <= F.lit(q1), F.lit(0))
         .when(F.col(col) <= F.lit(q2), F.lit(1))
         .otherwise(F.lit(2))
    )

feature_ts_expr = F.current_timestamp() if (run_ts is None or run_ts.strip()=="") else F.to_timestamp(F.lit(run_ts))


# 2. Load Gold sources

In [0]:
dim_plan = spark.table("cms_partd_gold.mart.dim_plan")
form_agg = spark.table("cms_partd_gold.mart.agg_plan_formulary_metrics")
net_agg  = spark.table("cms_partd_gold.mart.agg_plan_network_metrics")
price_agg= spark.table("cms_partd_gold.mart.agg_plan_pricing_metrics")
cost_agg= spark.table("cms_partd_gold.mart.agg_plan_cost_structure_metrics")

ins_cost = spark.table("cms_partd_gold.mart.agg_plan_insulin_cost_metrics")
ins_acc  = spark.table("cms_partd_gold.mart.agg_plan_insulin_access_metrics")
ins_frd  = spark.table("cms_partd_gold.mart.agg_plan_insulin_friendliness")

# if contract_year and contract_year.strip() != "":
#     dim_plan = dim_plan.where(F.col("contract_year") == F.lit(int(contract_year)))
    # optional: if your agg tables include contract_year, filter them too


# 3. Build feature_plan_summary (plan_key grain)

In [0]:
dim_plan  = spark.table("cms_partd_gold.mart.dim_plan")
form_agg  = spark.table("cms_partd_gold.mart.agg_plan_formulary_metrics")
net_agg   = spark.table("cms_partd_gold.mart.agg_plan_network_metrics")
price_agg = spark.table("cms_partd_gold.mart.agg_plan_pricing_metrics")

cost_struct = spark.table("cms_partd_gold.mart.agg_plan_cost_structure_metrics")

# insulin aggregates (if you built them)
# ins_cost = spark.table("cms_partd_gold.mart.agg_plan_insulin_cost_metrics")
# ins_acc  = spark.table("cms_partd_gold.mart.agg_plan_insulin_access_metrics")
ins_frd  = spark.table("cms_partd_gold.mart.agg_plan_insulin_friendliness")

base = (
    dim_plan
    .join(form_agg, on="plan_key", how="left")
    .join(net_agg, on="plan_key", how="left")
    .join(price_agg, on="plan_key", how="left")
    .join(cost_struct.drop("agg_ts"), on="plan_key", how="left")
    # .join(ins_cost, on="plan_key", how="left")
    # .join(ins_acc,  on="plan_key", how="left")
    .join(ins_frd.drop("gold_ts"),  on="plan_key", how="left")
    .withColumn("feature_ts", F.current_timestamp())
)

# Select a stable feature set (safe if some cols missing)
wanted = [
    # identifiers / categoricals
    "plan_key","contract_year","contract_type","state","county_code","snp","plan_suppressed_yn",
    # affordability inputs
    "premium","deductible",

    # formulary summary
    "formulary_breadth_ndc","formulary_breadth_rxcui","avg_tier_level",
    "pa_rate","st_rate","ql_rate","avg_restriction_count","restrictiveness_index",

    # network summary
    "network_size","preferred_retail_share","preferred_mail_share","in_area_share",
    "avg_floor_price","avg_brand_fee","avg_generic_fee",

    # pricing summary
    "avg_unit_cost","p90_unit_cost","p10_unit_cost","p90_p10_spread_unit_cost","priced_ndc_count",

    # cost structure (NEW – correct units)
    "ded_applies_rate","specialty_row_share",
    "pref_not_offered_share","pref_copay_share","pref_coinsurance_share",
    "pref_avg_copay_amt","pref_median_copay_amt","pref_p90_copay_amt",
    "pref_avg_coins_rate","pref_median_coins_rate","pref_p90_coins_rate",
    # tier-level
    *[f"pref_avg_copay_tier_{t}" for t in range(1,8)],
    *[f"pref_avg_coins_rate_tier_{t}" for t in range(1,8)],

    # insulin
    "insulin_min_copay_plan","insulin_max_copay_plan","insulin_exceeds_cap_rate","insulin_exceeds_cap_any_flag",
    "insulin_min_copay_30d","insulin_max_copay_30d","insulin_exceeds_cap_30d_flag",
    "insulin_pa_rate","insulin_st_rate","insulin_ql_rate","insulin_ndc_count","insulin_restrictiveness_index",
    "insulin_friendliness_score","insulin_cost_score","insulin_access_score","insulin_risk_flag",

    "feature_ts"
]

cols = [c for c in wanted if c in base.columns]
df_feat = base.select(*cols)

# Sanitize numerics for ML (avoid NaN/Inf)
non_numeric = {"plan_key","contract_year","contract_type","state","county_code","snp","plan_suppressed_yn","feature_ts"}
num_cols = [c for c in df_feat.columns if c not in non_numeric]

for c in num_cols:
    df_feat = df_feat.withColumn(c, F.col(c).cast("double"))
    df_feat = df_feat.withColumn(
        c,
        F.when(F.col(c).isNull(), None)
         .when(F.isnan(F.col(c)), None)
         .when(F.col(c) == float("inf"), None)
         .when(F.col(c) == float("-inf"), None)
         .otherwise(F.col(c))
    )

df_feat = df_feat.fillna({c: 0.0 for c in num_cols})

(df_feat.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema","true")
  .saveAsTable("cms_partd_gold.ml.feature_plan_summary"))


# 4. Build training_plan_labels

In [0]:
df = spark.table("cms_partd_gold.ml.feature_plan_summary")

def add_z(df, col):
    if col not in df.columns:
        return df.withColumn(f"{col}_z", F.lit(0.0))
    stats = df.agg(F.mean(col).alias("m"), F.stddev(col).alias("s")).collect()[0]
    m, s = stats["m"], stats["s"]
    if m is None or s is None or s == 0:
        return df.withColumn(f"{col}_z", F.lit(0.0))
    return df.withColumn(f"{col}_z", (F.col(col) - F.lit(m)) / F.lit(s))

# Z-scores for affordability components
for c in ["premium","deductible","pref_avg_copay_amt","pref_p90_copay_amt","pref_avg_coins_rate","pref_p90_coins_rate","ded_applies_rate"]:
    df = add_z(df, c)

# PCA-based Affordability Index
input_cols = ["premium_z", "deductible_z", "pref_avg_copay_amt_z", "pref_p90_copay_amt_z", "pref_avg_coins_rate_z", "pref_p90_coins_rate_z", "ded_applies_rate_z"]

assembler = VectorAssembler(inputCols=input_cols, outputCol="affordability_features")
pca = PCA(k=1, inputCol="affordability_features", outputCol="pca_features")
pipeline = Pipeline(stages=[assembler, pca])

model = pipeline.fit(df)
df = model.transform(df)

# Extract PC1 score (Costliness Index)
first_element = F.udf(lambda v: float(v[0]), T.DoubleType())
df = df.withColumn("affordability_index", first_element("pca_features"))

# Quantile buckets -> affordability_class (0=best, 2=worst)
q1, q2 = df.approxQuantile("affordability_index", [0.33, 0.66], 0.01)
df = df.withColumn(
    "affordability_class",
    F.when(F.col("affordability_index") <= F.lit(q1), 0)
     .when(F.col("affordability_index") <= F.lit(q2), 1)
     .otherwise(2)
)

# Restrictiveness class from restrictiveness_index if present
if "restrictiveness_index" in df.columns:
    r1, r2 = df.approxQuantile("restrictiveness_index", [0.33, 0.66], 0.01)
    df = df.withColumn(
        "restrictiveness_class",
        F.when(F.col("restrictiveness_index") <= F.lit(r1), 0)
         .when(F.col("restrictiveness_index") <= F.lit(r2), 1)
         .otherwise(2)
    )
else:
    df = df.withColumn("restrictiveness_class", F.lit(0))

# Network adequacy flag (rule-based)
if "preferred_retail_share" in df.columns and "in_area_share" in df.columns:
    df = df.withColumn(
        "network_adequacy_flag",
        F.when((F.col("preferred_retail_share") < 0.20) | (F.col("in_area_share") < 0.60), 1).otherwise(0)
    )
else:
    df = df.withColumn("network_adequacy_flag", F.lit(0))

# Insulin labels (if insulin cols exist)
if "insulin_exceeds_cap_any_flag" in df.columns and "insulin_exceeds_cap_rate" in df.columns:
    df = df.withColumn(
        "insulin_cap_violation_flag",
        F.when((F.col("insulin_exceeds_cap_any_flag") == 1) | (F.col("insulin_exceeds_cap_rate") > 0.05), 1).otherwise(0)
    )
else:
    df = df.withColumn("insulin_cap_violation_flag", F.lit(0))

if "insulin_restrictiveness_index" in df.columns and "insulin_pa_rate" in df.columns:
    df = df.withColumn(
        "insulin_access_risk_flag",
        F.when((F.col("insulin_restrictiveness_index") > 20) | (F.col("insulin_pa_rate") > 0.20), 1).otherwise(0)
    )
else:
    df = df.withColumn("insulin_access_risk_flag", F.lit(0))

# Optional insulin friendliness class if you want (0/1/2 by quantiles)
if "insulin_friendliness_score" in df.columns:
    i1, i2 = df.approxQuantile("insulin_friendliness_score", [0.33, 0.66], 0.01)
    df = df.withColumn(
        "insulin_friendliness_class",
        F.when(F.col("insulin_friendliness_score") <= F.lit(i1), 0)
         .when(F.col("insulin_friendliness_score") <= F.lit(i2), 1)
         .otherwise(2)
    )
else:
    df = df.withColumn("insulin_friendliness_class", F.lit(1))

(df.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema","true")
  .saveAsTable("cms_partd_gold.ml.training_plan_labels"))


In [0]:
spark.table("cms_partd_gold.mart.agg_plan_cost_structure_metrics") \
  .select(
      "plan_key",
      (F.col("pref_not_offered_share") + F.col("pref_copay_share") + F.col("pref_coinsurance_share")).alias("pref_share_sum")
  ).show(20, truncate=False)

spark.table("cms_partd_gold.ml.training_plan_labels") \
  .groupBy("affordability_class").count().orderBy("affordability_class").show()

spark.table("cms_partd_gold.ml.training_plan_labels") \
  .groupBy("insulin_cap_violation_flag").count().show()


# 5. Model prep

In [0]:
# from pyspark.sql import functions as F

df = spark.table("cms_partd_gold.ml.training_plan_labels")

# choose target
# label_col = "insulin_cap_violation_flag"   # (binary)
label_col = "affordability_class"        # (multiclass)

# optional: filter out suppressed plans
# if "plan_suppressed_yn" in df.columns:
#     df = df.filter((F.col("plan_suppressed_yn").isNull()) | (F.col("plan_suppressed_yn") != "Y"))


In [0]:
all_cols = df.columns

# Always exclude ids + timestamps + labels

exclude = {
    "plan_key"
    ,"feature_ts"
    ,"ingest_ts"
    ,"ingest_date"
    ,"source_file"
    ,"affordability_index"
    ,"network_adequacy_flag"
    ,"insulin_cap_violation_flag"
    ,"insulin_access_risk_flag"
    ,"plan_suppressed_yn"
    ,"insulin_friendliness_class"
    ,"affordability_class"
    ,"restrictiveness_class"
}
exclude.discard(label_col)  # keep label

cat_cols = [c for c in ["contract_type","state","snp"] if c in all_cols]

# Optional: include county_code but hashed (it is high-cardinality; safe with hashing)
if "county_code" in all_cols:
    cat_cols.append("county_code")

# Everything else numeric
num_cols = [c for c in all_cols if c not in exclude and c not in cat_cols and c != label_col]

# Keep only numeric-ish columns (drop accidentally non-numeric strings)
# We'll cast to double and fillna later.


In [0]:
data = df.select([label_col] + cat_cols + num_cols)

# cast numerics to double
for c in num_cols:
    data = data.withColumn(c, F.col(c).cast("double"))

# replace NaN/Inf with null then fill
for c in num_cols:
    data = data.withColumn(
        c,
        F.when(F.col(c).isNull(), None)
         .when(F.isnan(F.col(c)), None)
         .when(F.col(c) == float("inf"), None)
         .when(F.col(c) == float("-inf"), None)
         .otherwise(F.col(c))
    )

data = data.fillna({c: 0.0 for c in num_cols})

# categoricals: fill nulls with "UNKNOWN"
for c in cat_cols:
    data = data.withColumn(c, F.coalesce(F.col(c).cast("string"), F.lit("UNKNOWN")))


In [0]:
# from pyspark.ml import Pipeline
# from pyspark.ml.feature import FeatureHasher, VectorAssembler, StringIndexer
# from pyspark.ml.classification import LogisticRegression
# from pyspark.ml.evaluation import BinaryClassificationEvaluator

# train, test = data.randomSplit([0.8, 0.2], seed=42)

# # If label is string, index it. For binary int already, skip.
# label_is_numeric = dict(train.dtypes)[label_col] in ("int","bigint","double","float")

# stages = []

# # Hash categoricals into a compact vector
# # Tune numFeatures to control model size: 2^12=4096, 2^14=16384
# if len(cat_cols) > 0:
#     hasher = FeatureHasher(inputCols=cat_cols, outputCol="cat_features", numFeatures=(1<<14))
#     stages.append(hasher)
#     assembler_inputs = num_cols + ["cat_features"]
# else:
#     assembler_inputs = num_cols

# assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep")
# stages.append(assembler)

# lr = LogisticRegression(
#     featuresCol="features",
#     labelCol=label_col,
#     maxIter=50,
#     regParam=0.1,
#     elasticNetParam=0.0
# )
# stages.append(lr)

# pipe = Pipeline(stages=stages)

# model = pipe.fit(train)
# pred = model.transform(test)

# auc = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderROC").evaluate(pred)
# print("AUC:", auc)


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

data_mc = data.select([label_col] + cat_cols + num_cols)

train, test = data_mc.randomSplit([0.8, 0.2], seed=42)

lr_mc = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    family="multinomial",
    maxIter=80,
    regParam=0.1
)

stages = []
if len(cat_cols) > 0:
    stages.append(FeatureHasher(inputCols=cat_cols, outputCol="cat_features", numFeatures=(1<<14)))
    assembler_inputs = num_cols + ["cat_features"]
else:
    assembler_inputs = num_cols

stages.append(VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep"))
stages.append(lr_mc)

pipe = Pipeline(stages=stages)
model = pipe.fit(train)
pred = model.transform(test)

f1 = MulticlassClassificationEvaluator(labelCol=label_col, metricName="f1").evaluate(pred)
acc = MulticlassClassificationEvaluator(labelCol=label_col, metricName="accuracy").evaluate(pred)
print("F1:", f1, "Accuracy:", acc)


In [0]:
test.createOrReplaceTempView("view_test")

In [0]:
%sql
select contract_type, state, count(*)--affordability_class, count(*) 
from view_test
where affordability_class = 0
group by all